# 4 · How realistic is this market?

Realism here is a stated envelope rather than a score. Fourteen statistics
are measured against real-market bands, and what the model still gets wrong
is named rather than left for you to find. Five gaps are on the record, and
the list is measured rather than declared: a gap leaves it when a
measurement closes it.


In [1]:
import pretium as pt
from pretium import envelope as env
from pretium.facts import REAL_MARKETS, band_distance

print("preset certified :", env.PRESET)
print("certified horizon:", env.CERTIFIED_HORIZON_DAYS, "trading days")

preset certified : pt-v14
certified horizon: 252 trading days


## The panel

Measured at 30 seeds, 40 instruments, 252 days.

In [2]:
print(f"{'statistic':26s} {'measured':>10s} {'band':>20s}   verdict")
for name in REAL_MARKETS:
    lo, hi = REAL_MARKETS[name]
    v = env.CERTIFIED[name]
    ok = band_distance(v, lo, hi) == 0
    print(f"{name:26s} {v:10.4f} {f'[{lo:g}, {hi:g}]':>20s}   "
          f"{'in band' if ok else 'OUT'}")

n = sum(1 for k in REAL_MARKETS
        if band_distance(env.CERTIFIED[k], *REAL_MARKETS[k]) == 0)
print(f"\n{n} of {len(REAL_MARKETS)} in band")

statistic                    measured                 band   verdict
annualised_vol_pct            28.3103             [15, 36]   in band
excess_kurtosis               10.0043            [1.6, 41]   in band
return_acf1                    0.0114        [-0.08, 0.06]   in band
abs_return_acf1                0.0769         [0.02, 0.22]   in band
abs_return_acf5                0.0305         [0.01, 0.12]   in band
abs_return_acf20               0.0096        [-0.04, 0.08]   in band
cross_sectional_corr           0.2616         [0.08, 0.56]   in band
volume_abs_return_corr         0.5108         [0.46, 0.66]   in band
leverage_effect               -0.0258           [-0.16, 0]   in band
volume_change_acf1            -0.2794        [-0.32, -0.2]   in band
corr_asymmetry                -0.0018        [-0.25, 0.45]   in band
corr_asymmetry_lagged         -0.0327         [-0.2, 0.55]   in band
sector_excess_corr             0.2081         [0.11, 0.23]   in band
corr_persistence_acf1          0.1

## Nothing fails at one year, and the ruler changes at two

Every statistic is inside its band at the certified horizon: 14 of 14 above.
`volume_change_acf1` is among them, at -0.2794 against a band of -0.32 to
-0.2, and that row carried a gap of its own until the current default closed
it. What retired the gap is worth being precise about, because being in band
here was not enough on its own. The gap had already been rewritten around
the horizon -- the row held at one year and missed at two -- so holding at
one year left it standing. It came off the list only when the two-year miss
closed as well, which is what the refusal further down reports when it says
the model holds all 14 at 504 days against horizon-matched bands. That is
why the list below runs to five gaps rather than six.

The two-year ruler is a different ruler, not a stricter copy of the same
one. All fourteen bands are re-derived at 504 days and most come out
tighter, so the very same panel reads 14 of 14 against the 252-day ruler and
12 of 14 against the 504-day one without a single measurement changing.
`excess_kurtosis` is the case the cell prints: 6.700 sits well inside the
one-year band of 1.6 to 41 and outside the two-year band of 7.1 to 22. That
is a statement about the window the bands were derived on, not a two-year
failure -- a panel measured over two years is a different panel, and the
refusal printed further down reports that the model holds all 14 there
against horizon-matched bands.

## Scoring a panel

`envelope.score` reads a panel against the ruler for its own horizon. The
same numbers score differently at 252 and 504 days, which is the mistake
this function exists to prevent.


In [3]:
panel = {k: env.CERTIFIED[k] for k in REAL_MARKETS}

near = env.score(panel, horizon_days=252)
far = env.score(panel, horizon_days=504)

print(f"252-day ruler ({near['ruler']}): {near['in_band']}/{near['of']} in band")
print(f"504-day ruler ({far['ruler']}): {far['in_band']}/{far['of']} in band")
print()
k = "excess_kurtosis"
for label, s in (("252", near), ("504", far)):
    r = s["statistics"][k]
    print(f"  {k} @{label}d: {r['measured']:.3f} vs band {r['band']}"
          f"  -> {'in' if r['in_band'] else 'OUT'}")

252-day ruler (REAL_MARKETS): 14/14 in band
504-day ruler (REAL_MARKETS_504): 13/14 in band

  excess_kurtosis @252d: 10.004 vs band (1.6, 41.0)  -> in
  excess_kurtosis @504d: 10.004 vs band (7.1, 22.0)  -> in


`room_sd` gives the distance inside a band in that horizon's own seed noise.
A statistic barely inside is one seed away from being outside, and a plain
band check cannot distinguish the two.

In [4]:
rows = [(k, r["room_sd"]) for k, r in near["statistics"].items()
        if r["in_band"] and r["room_sd"] is not None]
for k, room in sorted(rows, key=lambda kv: kv[1]):
    flag = "  <- thin" if room < 0.5 else ""
    print(f"  {k:26s} {room:6.2f} sd inside{flag}")

  leverage_effect              0.34 sd inside  <- thin
  abs_return_acf5              0.37 sd inside  <- thin
  abs_return_acf1              0.60 sd inside
  return_acf1                  0.91 sd inside
  abs_return_acf20             1.06 sd inside
  volume_abs_return_corr       1.17 sd inside
  annualised_vol_pct           1.19 sd inside
  corr_persistence_acf1        1.30 sd inside
  corr_asymmetry_lagged        1.42 sd inside
  corr_asymmetry               1.54 sd inside
  cross_sectional_corr         1.67 sd inside
  sector_excess_corr           3.22 sd inside
  volume_change_acf1           3.97 sd inside
  excess_kurtosis              7.17 sd inside


## The gaps

Each gap names what it stops you concluding.

In [5]:
for gap in env.GAPS:
    print(f"* {gap.id}")
    print(f"    forbids: {gap.forbids}")

* horizon
    forbids: multi-year backtests, and anything keyed on volatility dynamics beyond one year
* decay-shape
    forbids: strategies whose edge depends on volatility memory beyond about lag 20 -- vol targeting and risk parity on a one-month or longer estimate
* scenario-magnitude
    forbids: sizing a scenario's impact rather than detecting it
* macro-range
    forbids: studying inflation regimes or policy crises from the endogenous economy alone
* roster-concentration
    forbids: inheriting this envelope for a sector-concentrated roster BEYOND one year -- at the certified horizon it now transfers


## Checking your own question

`check` refuses questions that fall outside the envelope, and every refusal
names the measurement behind it.

In [6]:
questions = [
    ("a one-year momentum study", dict(horizon_days=252,
                                       statistics=["return_acf1"])),
    ("a three-year study",        dict(horizon_days=756,
                                       statistics=["abs_return_acf1"])),
    ("volatility clustering decay", dict(horizon_days=252,
                                         statistics=["abs_return_acf20"])),
    ("a tech-only roster",        dict(horizon_days=252,
                                       sector_concentrated=True)),
]

for label, kwargs in questions:
    v = env.check(**kwargs)
    print(f"{label:32s} {'INSIDE' if v.inside else 'outside'}")
    if not v.inside:
        for reason in v.reasons:
            print(f"      {reason[:96]}")

a one-year momentum study        INSIDE
a three-year study               outside
      horizon 756d exceeds the certified 252d. At 504 days the model holds all 14 against horizon-matc
volatility clustering decay      outside
      abs_return_acf20 depends on the decay shape, which is a mechanism gap: log-log slope -0.953 agai
a tech-only roster               outside
      the roster is sector-concentrated, and certification was measured on a sector-balanced one. Re-m


## Choosing a preset

`pt-v14` is the default and what the envelope certifies at 252 days, as the
first cell printed. It has been the default since 2026-08-28; `pt-v12` held
the job before it, and every earlier preset stays selectable, so work
published against one of them keeps reproducing.

`envelope.regressions` names what a panel gives up against the shipped one.
Below, `pt-v3` gives up two rows to `pt-v14`: sector co-movement and
volume-change autocorrelation.

`pt-v12`, the previous default, reads 13 of 14 here, and the row it gives up
says more about sample size than about the preset. Three seeds is not
thirty. `corr_persistence_acf1` is one of the noisy rows: its band runs from
-0.19 to 0.54, wide because the statistic's spread across seeds is wide, and
the certified thirty-seed figure of 0.1771 sits 1.30 sd inside it in the
table above. A three-seed median of a statistic like that can land outside a
band the thirty-seed figure holds. Read `envelope.intervals` before trusting
any one run.


In [7]:
import statistics as _stats
from pretium import facts

small = tf.Universe.random(40, seed=111)

def panel_for(preset, seeds=(1, 2, 3)):
    model = tf.ModelParams.from_preset(preset)
    panels = [facts.measure(seed=s, universe=small, days=252, model=model)
              for s in seeds]
    return {k: _stats.median(p[k] for p in panels) for k in REAL_MARKETS}

for preset in ("pt-v10", "pt-v3"):
    p = panel_for(preset)
    s = env.score(p)
    print(f"{preset}:  {s['in_band']}/{s['of']} in band at 252d"
          f"   regressions vs shipped: {env.regressions(p) or 'none'}")


pt-v10:  13/14 in band at 252d   regressions vs shipped: ['corr_persistence_acf1']


pt-v3:  12/14 in band at 252d   regressions vs shipped: ['sector_excess_corr', 'volume_change_acf1']


Three seeds is a small sample; the published figures use thirty. Treat the
counts above as indicative. What matters is the shape of the trade, and
that `regressions` reports it rather than leaving you to count by hand.

**Take the default, `pt-v14`, unless you are reproducing work published
against an older preset -- and note that nothing here certifies a
multi-year question, whichever preset you pick.**

## Summary

- Realism is a set of measurements against bands, with the failures named.
- The certified horizon is 252 days, and `check` refuses beyond it.
- Good results here do not predict real returns.

Full documentation: <https://simoncoombes.github.io/pretium/>